# Assignment 06: A First-Principles Species Distribution Model

## BIO597 Spatial Analysis of Biodiversity

This assignment asks you to build and interpret a simple one-variable species distribution model without using an SDM package. You will choose one amphibian species, describe the environmental values at its occurrence records, compare those values with background environments, create a suitability curve, and project that curve across Maine.

## Deliverable

Submit a completed notebook with code, maps, plots, and short written interpretations. Your goal is to show that you understand the moving parts of an SDM before we use more automated modeling tools.

## Learning goals

By the end of this assignment, you should be able to:

* describe the difference between occurrence, background, and absence data
* extract one raster predictor at species occurrence records
* compare used environments with available environments
* create a simple environmental suitability score
* project a suitability score across raster space
* explain why a conceptual SDM is not the same as a validated prediction model

## Data

Use:

* `Amphibians_ME/Amphibians_ME.csv` for occurrence records
* `/home/jovyan/shared-readwrite/bioclim/` for one environmental variable of your choice that is NOT mean annual temperature
* `../labs/ME_ShapeFiles/Maine_State_Boundary.shp` for the Maine boundary

You may use the same species as the lab or choose a different species with at least 100 usable records.

## 1. Import packages

Import `pandas`, `geopandas`, `rioxarray`, `xarray`, `numpy`, `matplotlib.pyplot`, and `Path`.

In [ ]:
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

## 2. Load and combine the Maine boundary

Read the Maine state boundary shapefile, inspect its CRS and shape, and dissolve it to one row.

In [ ]:
boundary_path = "../labs/ME_ShapeFiles/"
maine_boundary = gpd.read_file(os.path.join(boundary_path, "Maine_State_Boundary.shp"))
print(maine_boundary.shape)
print(maine_boundary.crs)

maine_boundary = maine_boundary.dissolve()
maine_boundary.explore()

## 3. Open and clip environmental data

Select one environmental variable of your choosing that is NOT mean annual temperature. You can choose one you think will work well or one you think is less important for the study organism. Open the variable .tif file with `masked=True`, clip it to Maine, and remove the band dimension with `squeeze()`.


In [ ]:
bioclim_path = Path("C:/Users/elisabeth.maxwell/Data/wc2.1_2.5m_bio")
bio1 = rxr.open_rasterio(
    bioclim_path / "wc2.1_2.5m_bio_1.tif",
    masked=True,
)

bio1_maine = bio1.rio.clip(
    maine_boundary.geometry,
    maine_boundary.crs,
    drop=True,
    from_disk=True,
).squeeze()

if float(bio1_maine.max()) > 100:
    bio1_maine_c = bio1_maine / 10
else:
    bio1_maine_c = bio1_maine

bio1_maine_c.name = "annual_mean_temperature_c"
bio1_maine_c

## 4. Map your environmental variable

Create a map of the clipped temperature raster. Include a title.

In [ ]:
bio1_maine_c.plot(cmap="coolwarm")
plt.title("Annual mean temperature in Maine")
plt.show()

## 5. Load and prepare amphibian occurrence records

Read the tab-delimited amphibian file. Keep `gbifID`, `species`, longitude, latitude, coordinate uncertainty, and year. Convert coordinates to numeric values and remove records missing species or coordinates.

In [ ]:
amphibians = pd.read_csv(
    "../assignments/Amphibians_ME/Amphibians_ME.csv",
    sep="\t",
    low_memory=False,
)

columns_to_keep = [
    "gbifID",
    "species",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "year",
]

amphibians = amphibians[columns_to_keep].copy()
amphibians["decimalLongitude"] = pd.to_numeric(amphibians["decimalLongitude"], errors="coerce")
amphibians["decimalLatitude"] = pd.to_numeric(amphibians["decimalLatitude"], errors="coerce")

# Simple cleaning
amphibians = amphibians.dropna(subset=["species", "decimalLongitude", "decimalLatitude"]).copy()

# Reduce spatial imprecision (The "tulipifera-ike" filter!)
amphibians = amphibians[amphibians["coordinateUncertaintyInMeters"] < 200]

## 6. Choose one species

Display the most common species, then choose one species with at least 100 records. Store its scientific name as `target_species`.

In [ ]:
amphibians["species"].value_counts().head(12)

In [ ]:
target_species = "Lithobates palustris"

presence = amphibians.loc[amphibians["species"] == target_species].copy()
print(target_species)
print(presence.shape)
presence.head(5)

**Why did you choose this species?** Include one sentence about record count and one sentence about why it may or may not be appropriate for this simple model.
- *Lithobates palustris* has 1049 records available (after filtering for missing latitude/longitude, and filtering for spatial uncertainty). While *L. palustris* is in the top 10 for most number of records available in this dataset, 1049 is not a very high number of occurrence records considering the size of the state of Maine and potential spatial heterogeneity. 

## 7. Create an occurrence GeoDataFrame

Use longitude and latitude to create point geometry. Assign `EPSG:4326`. Confirm that the occurrence CRS and raster CRS are compatible.

In [ ]:
presence_gdf = gpd.GeoDataFrame(
    presence,
    geometry=gpd.points_from_xy(
        presence["decimalLongitude"],
        presence["decimalLatitude"],
    ),
    crs="EPSG:4326",
)

print("Occurrence CRS:", presence_gdf.crs)
print("Raster CRS:", bio1_maine_c.rio.crs)

presence_gdf.explore()

**CRS check:** Explain why coordinate systems matter before extracting raster values at points.
-  A CRS defines how coordinates project the raster on Earth. If the CRS is not aligned, then the data will not match up and the analysis will look at the wrong locations. Because we've confirmed the CRS matches in Step 7, we can proceed to extracting the values in Step 8.

## 8. Extract environment at occurrence points

Create `xarray.DataArray` objects for point longitude and latitude. Use `.sel(..., method="nearest")` to extract BIO1 values at occurrence records. Add the extracted values to a column named `bio1_c`.

In [ ]:
point_x = xr.DataArray(presence_gdf.geometry.x.values, dims="record")
point_y = xr.DataArray(presence_gdf.geometry.y.values, dims="record")

presence_temp = bio1_maine_c.sel(
    x=point_x,
    y=point_y,
    method="nearest",
)

presence_gdf["bio1_c"] = presence_temp.values
presence_gdf = presence_gdf.dropna(subset=["bio1_c"]).copy()

point_x = xr.DataArray(presence_gdf.geometry.x.values, dims="record")
point_y = xr.DataArray(presence_gdf.geometry.y.values, dims="record")

presence_gdf[["species", "decimalLongitude", "decimalLatitude", "bio1_c"]].head()

## 9. Summarize occurrence environment

Calculate the count, mean, standard deviation, minimum, and maximum of `bio1_c` at occurrence records.

In [ ]:
presence_mean = presence_gdf["bio1_c"].mean()
presence_sd = presence_gdf["bio1_c"].std()

print(f"Number of usable records: {len(presence_gdf)}")
print(f"Mean occurrence temperature: {presence_mean:.2f} C")
print(f"SD of occurrence temperature: {presence_sd:.2f} C")
print(f"Minimum occurrence temperature: {presence_gdf['bio1_c'].min():.2f} C")
print(f"Maximum occurrence temperature: {presence_gdf['bio1_c'].max():.2f} C")

**Interpretation:** In two or three sentences, describe the environmental range represented by your species records.
- BIO1 data represents the Annual Mean Temperature. For the study area encompassing *L. palustris* occurrence records, the average Annual Mean Temperature was 6.61C with a standard deviation of 0.86C. The minimum Annual Mean Temperature observed within the study area was 2.59C and the maximum Annual Mean Temperature observed was 8.79C. 

## 10. Sample background environment

Create a random sample of up to 5,000 valid cells from Maine for your environmental variable. Use a random seed so your results are reproducible.

In [ ]:
rng = np.random.default_rng(597)

valid_temperature = bio1_maine_c.values
valid_temperature = valid_temperature[np.isfinite(valid_temperature)]

background_temp = rng.choice(
    valid_temperature,
    size=min(5000, valid_temperature.size),
    replace=False,
)

print(f"Background sample size: {len(background_temp)}")
print(f"Mean background temperature: {background_temp.mean():.2f} C")
print(f"SD background temperature: {background_temp.std():.2f} C")

## 11. Compare occurrence and background environments

Make overlapping histograms of occurrence environment and background environment. Use density-scaled histograms so the shapes can be compared even though the sample sizes differ.

In [ ]:
plt.hist(background_temp, bins=30, alpha=0.5, density=True, label="Background")
plt.hist(presence_gdf["bio1_c"], bins=30, alpha=0.6, density=True, label="Occurrences")
plt.axvline(presence_mean, color="black", linestyle="--", label="Occurrence mean")
plt.xlabel("Annual mean temperature (C)")
plt.ylabel("Density")
plt.title(f"Temperature environments for {target_species}")
plt.legend()
plt.show()

**Interpretation:** Does your species appear to use a narrow or broad subset of available environmental space? Explain using your plot.
- The Ocurrence distribution is shifted right, relative to the background temperature distribution. The mean occurrences falls at ~6.5C, which is almost 2 degrees C higher than the mean background temperature (calculated in Step 10). This histogram indicates that the *L. palustris* observations are associate with warmer temperatures.

## 12. Define a suitability function

Write a function that takes `observed`, `optimum`, and `tolerance` and returns:

```python
np.exp(-0.5 * ((observed - optimum) / tolerance) ** 2)
```

This creates a score near 1 when observed environment is close to the optimum and closer to 0 as temperature gets farther away.

In [ ]:
def environmental_suitability(observed, optimum, tolerance):
     z = (observed - optimum) / tolerance
     return np.exp(-0.5 * z**2)

## 13. Plot your suitability curve

Use the mean occurrence environmental value as the optimum and the standard deviation of occurrence as the tolerance. Plot suitability across the full range of Maine BIO1 values.

In [ ]:
## Caculated in Step 9: presence_mean = presence_gdf["bio1_c"].mean()
## Caculated in Step 9: presence_sd = presence_gdf["bio1_c"].std()

x_temp = np.linspace(
    float(np.nanmin(bio1_maine_c.values)),
    float(np.nanmax(bio1_maine_c.values)),
    200,
)

suitability_curve = environmental_suitability(
    x_temp,
    optimum=presence_mean,
    tolerance=presence_sd,
)

plt.plot(x_temp, suitability_curve)
plt.axvline(presence_mean, color="black", linestyle="--", label="Occurrence mean")
plt.xlabel("Annual mean temperature (C)")
plt.ylabel("Suitability score")
plt.ylim(0, 1.05)
plt.title("A hand-built temperature suitability curve")
plt.legend()
plt.show()

**Interpretation:** What value receives the highest suitability score? What does the tolerance control?
- The highest suitability is at the "optimum", where the mean temperature of occurrence records occurs. The tolerance is defined by the standard deviation and controls the width of the curve. 

## 14. Project suitability across Maine

Apply your suitability function to every cell in the environmental raster. Keep missing raster cells missing.

In [ ]:
suitability = xr.apply_ufunc(
    environmental_suitability,
    bio1_maine_c,
    presence_mean,
    presence_sd,
)

suitability = suitability.where(np.isfinite(bio1_maine_c))
suitability = suitability.rio.write_crs(bio1_maine_c.rio.crs)
suitability.name = "temperature_suitability"
suitability

## 15. Map suitability

Create a continuous suitability map with values from 0 to 1.

In [ ]:
suitability.plot(vmin=0, vmax=1, cmap="viridis")
plt.title(f"Temperature-based suitability for {target_species}")
plt.show()

## 16. Overlay occurrence points

Plot your suitability raster and overlay the occurrence points. This is a visual check, not a formal validation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
suitability.plot(ax=ax, vmin=0, vmax=1, cmap="viridis")
presence_gdf.plot(ax=ax, markersize=6, color="white", edgecolor="black", linewidth=0.2)
ax.set_title(f"Occurrences over temperature suitability for {target_species}")
plt.show()

**Visual check:** Do occurrence points generally fall in areas with higher predicted suitability? Describe any mismatch you see.
 - Yes, visually, the occurrence points are predominately located within the band of higher suitability. However, there are also points located in areas of low predicted suitability. In general, the occurrence points that fall in areas of low predicted suitability are generally close to areas with high suitability. For example: there are no occurrences observed at the very northern edge of the state. 

## 17. Compare occurrence and background suitability scores

Extract suitability at occurrence records. Also calculate suitability for your background environmental values. Compare the two distributions with histograms.

In [ ]:
presence_suitability = suitability.sel(
    x=point_x,
    y=point_y,
    method="nearest",
).values

background_suitability = environmental_suitability(
    background_temp,
    optimum=presence_mean,
    tolerance=presence_sd,
)

print(f"Mean occurrence suitability: {np.nanmean(presence_suitability):.3f}")
print(f"Mean background suitability: {np.nanmean(background_suitability):.3f}")

plt.hist(background_suitability, bins=30, alpha=0.5, density=True, label="Background")
plt.hist(presence_suitability, bins=30, alpha=0.6, density=True, label="Occurrences")
plt.xlabel("Suitability score")
plt.ylabel("Density")
plt.title("Suitability scores at occurrences and background cells")
plt.legend()
plt.show()

**Interpretation:** Did your model score occurrence records higher than background locations? What would it mean if it did not?
- Yes, the mean occurrence suitability is 0.776 and the mean background suitability at 0.349. This indicates that the suitability generated from occurrence records is accurately scoring high than general Maine background. In the histogram, the occurrences are clustered towards the right (high suitability), and the background is clustered to the left (low suitability). 
- If the suitability scores were the same for both, it could indicate that the sample area was small enough that you were only evaluating areas that were high suitability. In order to accurately determine habitat suitability, you need to ensure that the analysis area includes geographical areas with low suitability. Other reasons that you would see model overlap include: predictor variable is not a strong driver (example: temperature), or there is a bug in the code (example: mismatched CRS or wrong units).

## 18. Change the tolerance

Create two additional suitability rasters: one with half the observed standard deviation and one with twice the observed standard deviation. Map all three side by side.

In [ ]:
narrow_suitability = xr.apply_ufunc(
    environmental_suitability,
    bio1_maine_c,
    presence_mean,
    presence_sd / 2,
).where(np.isfinite(bio1_maine_c))

broad_suitability = xr.apply_ufunc(
    environmental_suitability,
    bio1_maine_c,
    presence_mean,
    presence_sd * 2,
).where(np.isfinite(bio1_maine_c))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
narrow_suitability.plot(ax=axes[0], vmin=0, vmax=1, cmap="viridis")
axes[0].set_title("Narrow tolerance")
suitability.plot(ax=axes[1], vmin=0, vmax=1, cmap="viridis")
axes[1].set_title("Observed SD tolerance")
broad_suitability.plot(ax=axes[2], vmin=0, vmax=1, cmap="viridis")
axes[2].set_title("Broad tolerance")
plt.tight_layout()
plt.show()

**Interpretation:** How does tolerance affect the predicted distribution? Which tolerance seems most defensible for your species, and why?
- Yes, increasing the tolerace results in a wider (broader) band for temperature suitability. Based on our plot of occurrence records, the broad tolerance would better capture the observed temperature tolerance for *L palustris* in Maine.

## 19. Threshold the continuous map

Convert your continuous suitability map to a binary suitable/unsuitable map using a threshold of your choice. Report the fraction of Maine climate cells labeled suitable.

In [ ]:
threshold = 0.4
suitable_binary = suitability >= threshold
suitable_binary = suitable_binary.where(np.isfinite(bio1_maine_c))

suitable_binary.plot(cmap="Greens")
plt.title(f"Cells with suitability >= {threshold}")
plt.show()

suitable_fraction = float(suitable_binary.mean(skipna=True))
print(f"Fraction of Maine climate cells labeled suitable: {suitable_fraction:.2%}")

**Threshold justification:** Why did you choose this threshold? How would the map change with a lower or higher threshold?
- I chose 0.4 as a threshold to better align with the pattern of occurrence records.
- The map gets more narrow with a higher threshold and more broad with a lower threshold. A broad tolerance predicts more suitable area and a narrow tolerance predicts less suitabile area.

## 20. Save your outputs

Save your continuous suitability raster as `Assignment06_suitability.tif`. Save a CSV containing your occurrence records and extracted BIO1 values as `Assignment06_presence.csv`.

In [ ]:
suitability.rio.to_raster("Assignment06_temperature_suitability.tif")

presence_table = presence_gdf.drop(columns="geometry")
presence_table["environmental_suitability"] = presence_suitability
presence_table.to_csv("Assignment06_presence_temperature_suitability.csv", index=False)

## Final reflection

Answer each question in two or three sentences.

1. In this exercise, what information came from occurrence records and what information came from the environmental raster?
 - The occurrence records tell us the location of where this species was observed. The environmental raster tells us about the background environmental conditions throughout the study area (in this specific case: Annual Mean Temperature). 
2. Why did we compare occurrence environment with background environment?
 - Occurrences tells us which environments are being used by the species. Background tells us the available environments across the region where we are making predictions. 
3. Why is this suitability score not the same as probability of presence?
- Suitability is a relative score based on similarity to the model. In this model, a high suitability score means "similar to known occurrence environments." Probabiliy of presence requires stronger statistical assumptions and usually validation data.

4. Name two ecological or sampling processes ignored by this one-variable model.
 - This model does not account for uneven sampling effort or precipitation patterns or minimum annual temperature.
5. What would a full SDM package add beyond this first-principles workflow?
 - SDM packages allow for inclusion of multiple predictors and interactions, flexible response curves, different sampling designs and background strategies, and include tools for cross-validation, uncertainty quantification, model comparison, and reproducable workflows. 

## Submit your work

Run the notebook from top to bottom. Make sure all maps, plots, saved files, and written answers are complete. Then commit and push the completed notebook to your class GitHub repository.

```bash
git add docs/assignments/Assignment-06-FirstPrinciplesSDM.ipynb
git commit -m "Complete first principles SDM assignment"
git push
```